In [ ]:
import struct # struct模块，用于解析二进制数据（如将 bytes 转为整数）
#%run ./3.FullyConnected.ipynb  #原始复杂的全连接网络
%run ./4.VectorFCNN.ipynb  #向量化的全连接网络
from datetime import datetime

#数据加载器基类
class Loader(object):
    def __init__(self,path,count):
        """初始化加载器。path数据文件路径。count文件中样本个数"""
        self.path = path
        self.count = count
    def get_file_content(self):
        """读取文件内容。"""
        f = open(self.path,'rb')
        content = f.read()
        f.close()
        return content  #返回bytes对象
    def to_int(self, byte):
        """将 unsigned byte 字符转为整数，兼容 int 或 bytes"""
        if isinstance(byte, int): # 在 Python 3中，从 bytes对象通过索引取值时，得到的是 int（0~255），而不是长度为 1 的 bytes
            return byte # 如果已经是整数（Python 3 中 bytes 索引返回 int）
        else:
            return struct.unpack('B', byte)[0] # 解包为无符号字符（0~255）

#图像数据加载器
class ImageLoader(Loader):
    def get_picture(self,content,index):
        """从文件中获得图像。content: 整个文件的bytes内容。index: 图片索引（0 开始）。
        返回一个二维列表，每个元素是0~255的像素值"""
        start = index*28*28+16 # 计算该图片在文件中的起始偏移量（前16字节为魔数和维度信息）
        picture = [] # 存储整张图片的二维列表
        for i in range(28):
            picture.append([])
            for j in range(28):
                picture[i].append(self.to_int(content[start+i*28+j]))
        return picture
    def get_one_sample(self,picture):
        """将一张图片（二维列表）转换为一个一维样本向量（长度为784）"""
        sample = []
        for i in range(28):
            for j in range(28):
                sample.append(picture[i][j])
        return sample
    def load(self):
        """加载整个图像文件，返回所有样本的向量列表（每个样本是长度为784的列表）"""
        content = self.get_file_content()
        data_set = []
        for index in range(self.count):
            data_set.append(self.get_one_sample(self.get_picture(content,index)))
        return data_set

#标签数据加载器
class LabelLoader(Loader):
    def load(self):
        """加载数据文件，获得全部样本的标签向量"""
        content = self.get_file_content()
        labels = []
        for index in range(self.count): # 标签文件的前8字节为魔数和维度，第 index+8 字节是第 index 个标签的原始值（0~9）
            labels.append(self.norm(content[index+8]))
        return labels
    def norm(self,label):
        """将原始标签值（整数0~9）转换为10维one-hot 向量，目标位用0.9，其他位用0.1"""
        label_vec = []  # 存储10维向量
        label_value = self.to_int(label)  # 将 bytes 或 int 转为整数
        for i in range(10):  # 遍历10个类别
            if i == label_value:  # 如果是正确的类别，设为0.9，避免饱和
                label_vec.append(0.9)
            else:
                label_vec.append(0.1)
        return label_vec
        
def get_training_data_set():
    """获得训练数据"""
    image_loader = ImageLoader(r'.\MNIST_data\train-images-idx3-ubyte\train-images.idx3-ubyte',60000)
    label_loader = LabelLoader(r'.\MNIST_data\train-labels-idx1-ubyte\train-labels.idx1-ubyte',60000)
    return image_loader.load(),label_loader.load() 
    
def get_test_data_set():
    """获得测试数据集"""
    image_loader = ImageLoader(r'.\MNIST_data\t10k-images-idx3-ubyte\t10k-images.idx3-ubyte',10000)
    label_loader = LabelLoader(r'.\MNIST_data\t10k-labels-idx1-ubyte\t10k-labels.idx1-ubyte',10000)
    return image_loader.load(),label_loader.load()

In [ ]:
# 网络的输出是一个10维向量，这个向量第个n (从0开始编号)元素的值最大，那么就是网络的识别结果。
def get_result(vec):
    max_value_index = 0
    max_value = 0
    for i in range(len(vec)):
        if vec[i] > max_value:
            max_value = vec[i]
            max_value_index = i
    return max_value_index

In [ ]:
# 我们使用错误率来对网络进行评估，下面是代码实现：
def evaluate(network, test_data_set, test_labels):
    """评估网络在测试集上的错误率（错误样本数 / 总样本数）"""
    error = 0   # 错误计数
    total = len(test_data_set)   # 总测试样本数
    for i in range(total):       # 遍历每个测试样本
        label = get_result(test_labels[i])
        predict = get_result(network.predict(test_data_set[i]))  # 网络预测标签
        if label != predict:      # 如果预测错误，错误数加1
            error += 1 
    return float(error) / float(total)

In [ ]:
# 每训练10轮，评估一次准确率，当准确率开始下降时终止训练。下面是代码实现：
def train_and_evaluate():
    """主训练函数：每次训练1个epoch，每10个epoch评估一次测试错误率，如果错误率比上一次评估上升则停止训练（早停）"""
    last_error_ratio = 1.0  # 上一次评估的错误率，初始为1（100%）
    epoch = 0
    train_data_set, train_labels = get_training_data_set()
    test_data_set, test_labels = get_test_data_set()
    network = Network([784, 300, 10])
    while True:  # 无限循环，直到满足早停条件
        epoch += 1
        network.train(train_labels, train_data_set, 0.3, 1) # 学习率固定为0.3，epoch参数为1（即只训练1轮），用外面设定的epoch来计数，用早停法结束，而不是提前设置好训练轮数
        print('%s epoch %d finished' % (datetime.now(), epoch)) 
        if epoch % 10 == 0:
            error_ratio = evaluate(network, test_data_set, test_labels)
            print('%s after epoch %d, error ratio is %f' % (datetime.now(), epoch, error_ratio))
        if error_ratio > last_error_ratio:
            break
        else:
            last_error_ratio = error_ratio
        
if __name__ == '__main__':
    train_and_evaluate()